# 05 · §7.5 — Painéis qualitativos de retrieval

**É só rodar tudo (Run All).** Sem variáveis de ambiente, sem editar caminhos.

O notebook:
1. lê a `WANDB_API_KEY` do `.env` (via `qual_utils`/`analysis_utils`);
2. escolhe, do `results.csv`, os runs de cada célula — **Graph-RKD** (headline
   `mds`, N=4, seed 0) vs o **baseline clássico mais forte** do dataset;
3. resolve o checkpoint de cada run (procura em `experiments_local/`, senão
   **baixa o artefato de modelo do W&B** — TTL 30 dias);
4. puxa o *test split* do S3 (cacheado) e embute com o student;
5. gera 1 figura por célula em `figures/fig_qual_<dataset>_<teacher>.pdf`.

Cada figura: por linha uma *query*; colunas = `[query | top-k Graph-RKD | top-k baseline]`.
Borda **verde** = mesma classe da query, **vermelha** = classe diferente. Mostro
acertos *e* erros do top-1 do Graph-RKD (modos de erro), não qualidade de ranking
— isso é papel das métricas (notebooks 01–04).

> Precisa de `torch`+`torchvision` e acesso ao W&B. Usa GPU se houver, senão CPU
> (mais lento). Rode **depois** do `00` (que gera o `results.csv`).

In [ ]:
# --- setup: importa qual_utils (carrega torch/torchvision + o .env) ---
import os, sys
sys.path.insert(0, os.getcwd())
import pandas as pd
import qual_utils as qu

df = pd.read_csv("results.csv")
print("device:", qu.DEVICE, "| runs no results.csv:", len(df))

## Inputs das análises (já preenchidos)

Baseline mais forte por dataset (§7.5): em **Cars** o RKD clássico *piora* → o
baseline é `triplet_only`; em **CUB** o `rkd_angle` (RKD-A) é o melhor clássico.
Graph-RKD sempre na config headline (`mds`, N=4). Ajuste só se quiser outras células.

In [ ]:
# ---- inputs (edite só se quiser outras células/config) ----
GRAPH_METHOD, GRAPH_N = "mds", 4          # config headline do Graph-RKD (§7.1)
TOPK = 5                                   # vizinhos por query
N_SUCCESS, N_FAIL = 3, 3                   # acertos + erros do top-1 do Graph-RKD

BASELINE_BY_DATASET = {"cars196": "triplet_only", "cub200": "rkd_angle"}

# uma figura por célula (dataset, teacher)
CELLS = [
    ("cars196", "resnet18"),
    ("cars196", "convnext_tiny"),
    ("cub200",  "resnet18"),
    ("cub200",  "convnext_tiny"),
]

In [ ]:
# ---- roda todas as células e salva as figuras ----
paths = []
for dataset, teacher in CELLS:
    baseline = BASELINE_BY_DATASET[dataset]
    try:
        p = qu.run_cell(df, dataset, teacher, baseline,
                        graph_method=GRAPH_METHOD, graph_N=GRAPH_N,
                        topk=TOPK, n_success=N_SUCCESS, n_fail=N_FAIL,
                        outdir="figures")
        if p:
            paths.append(p)
    except Exception as e:
        print(f"[erro] {dataset}/{teacher}: {type(e).__name__}: {e}")

import matplotlib.pyplot as plt
plt.show()
print("\nfiguras salvas:")
for p in paths:
    print("  -", p)